# Phase D: Temporal Prediction Investigation

This notebook performs development-only causal temporal analysis. At time t, all input columns are at t or earlier; the response is shifted to a future timestamp. It does not tune or fit a future-temperature ML model.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
if not (project_root / 'src').exists():
    project_root = Path.cwd()
sys.path.insert(0, str(project_root))

from src.data_preprocessing import chronological_split, load_and_validate_data
from src.temporal_features import (
    categorical_future_target_summary, create_lagged_frame, create_temporal_figures,
    evaluate_persistence_baseline, future_temperature_autocorrelation, numeric_lag_correlations,
)

data = load_and_validate_data(project_root / 'EV_Battery_Charging_TR_Dataset_with_Notes (1).csv')
development_data, final_test_data = chronological_split(data)
# Do not use final_test_data for exploratory feature decisions in this phase.
print('Development rows:', len(development_data))
print('Final holdout rows preserved:', len(final_test_data))

In [ ]:
intervals = development_data['Timestamp'].diff().dropna()
print('Observed interval counts:')
print(intervals.value_counts().sort_index())
autocorrelation = future_temperature_autocorrelation(development_data)
autocorrelation

In [ ]:
# Features at t and earlier; target is MaxTemp_C at t+5 minutes.
five_minute_frame = create_lagged_frame(development_data, horizon_minutes=5)
five_minute_frame[[
    'Timestamp', 'target_timestamp', 'MaxTemp_C_lag_0', 'MaxTemp_C_lag_5', 'MaxTemp_C_future_5m'
]].head()

In [ ]:
lag_correlations = numeric_lag_correlations(development_data, horizon_minutes=5)
lag_correlations.pivot(index='feature', columns='lag_minutes', values='correlation_with_future_max_temp')

In [ ]:
categorical_summary = categorical_future_target_summary(development_data, horizon_minutes=5)
categorical_summary.query('lag_minutes == 0')

In [ ]:
persistence_folds, persistence_summary = evaluate_persistence_baseline(development_data)
persistence_summary

In [ ]:
figure_paths = create_temporal_figures(development_data, project_root / 'results' / 'figures', horizon_minutes=5)
[str(path.relative_to(project_root)) for path in figure_paths]

## Scope boundary

The correlation and persistence results are exploratory. A lagged-feature ML model should only be built after confirming that the available signals have meaningful temporal predictive value and that the final holdout remains untouched.